# Day 3 · 실습 1. RNN 언어모델의 학습과 다음 토큰 예측

**연결 이론:** 이론 1의 Hidden State·출력 Shape, 이론 2의 입력·정답 Shift·Logits·Cross-Entropy·BPTT.

**이번 실습:** A형에서 주어에 따라 음료를 예측하는 RNN을 학습하고, B형 1단계에서는 입력·정답과 순전파를 완성합니다. B형 2단계에서는 **시간과 날씨 두 조건**에 따른 다음 토큰 예측·평가 함수를 직접 구현합니다.

**진행:** A형 18분 → B형 1단계 10분 → B형 2단계 20분 → 결과 확인 2분.

## A형 · 강사와 함께 구현하기

주어가 달라지면 '항상' 뒤에 등장할 음료가 달라지는 문장을 학습합니다. 마지막에는 같은 위치의 입력 토큰을 사용하더라도 앞 문맥에 따라 예측이 달라지는지 확인합니다.

### A-1. 라이브러리와 실행 설정

PyTorch와 데이터 로더를 준비하고 실행 조건을 고정합니다.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# 작은 수업용 모델이므로 CPU 스레드 하나로 실행합니다.
torch.set_num_threads(1)
torch.manual_seed(42)
print("PyTorch:", torch.__version__)


### A-2. 학습 문장과 다음 토큰 정답 만들기

주어 4개와 시간 표현 4개로 문장 16개를 만들고, EOS를 붙여 입력·정답을 한 칸 이동시킵니다. 마지막 출력에서 문장 수, 첫 입력·정답, Tensor Shape를 확인합니다.

In [ ]:
# 수업용 규칙입니다. 주어에 따라 '항상' 다음에 나오는 음료가 달라집니다.
preferences = {"나는": "커피를", "너는": "우유를", "우리는": "주스를", "친구는": "차를"}
time_words = ["아침에", "점심에", "저녁에", "밤에"]
sentences_a = [
    f"{subject} {when} 항상 {drink} 마신다"
    for subject, drink in preferences.items()
    for when in time_words
]

# 모든 문장 끝에 <EOS>를 붙여 마지막 단어 다음에는 종료 토큰을 학습합니다.
tokens_a = [sentence.split() + ["<EOS>"] for sentence in sentences_a]
vocab_a = sorted({token for sentence in tokens_a for token in sentence})
token_to_id_a = {token: index for index, token in enumerate(vocab_a)}

# nn.Embedding의 입력은 실수 벡터가 아니라 정수 토큰 ID입니다.
encoded_a = torch.tensor(
    [[token_to_id_a[token] for token in sentence] for sentence in tokens_a],
    dtype=torch.long,
)

# 원본: [나는, 아침에, 항상, 커피를, 마신다, <EOS>]
# 입력: 마지막 토큰을 제외 / 정답: 첫 토큰을 제외 -> 각 위치에서 다음 토큰 예측
x_a = encoded_a[:, :-1]
y_a = encoded_a[:, 1:]

# 입력과 정답을 같은 인덱스로 묶어, 섞어도 문장별 대응이 유지됩니다.
loader_a = DataLoader(TensorDataset(x_a, y_a), batch_size=5, shuffle=True)

print("문장 수 / 어휘 수:", len(sentences_a), len(vocab_a))
print("첫 입력:", [vocab_a[index] for index in x_a[0].tolist()])
print("첫 정답:", [vocab_a[index] for index in y_a[0].tolist()])
print("입력 / 정답 Shape:", tuple(x_a.shape), tuple(y_a.shape))


### A-3. RNN 언어모델 구성

Embedding → RNN → Linear를 연결합니다. RNN의 모든 시점에 대해 어휘별 Logits가 출력되도록 구성합니다.

In [ ]:
class RNNLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 16, hidden_size: int = 32):
        super().__init__()
        # 토큰 ID 하나를 embedding_dim개의 실수로 바꿉니다.
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        # 입력 축을 (문장 수, 시퀀스 길이, 임베딩 차원) 순서로 사용합니다.
        # 단방향 RNN이므로 현재 위치의 상태는 미래 토큰을 보지 않습니다.
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        # 각 시점의 Hidden State를 어휘 수만큼의 점수로 바꿉니다.
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        # (B, T) -> (B, T, 16): 각 토큰 ID에 해당하는 임베딩을 조회합니다.
        embedded = self.embedding(token_ids)
        # (B, T, 16) -> (B, T, 32): 모든 시점의 Hidden State를 받습니다.
        # 초기 상태를 생략하면 0으로 시작합니다. 별개의 문장 배치에 이전 배치 상태를 넘기지 않습니다.
        states, _ = self.rnn(embedded)
        # (B, T, 32) -> (B, T, V): 마지막 시점만이 아니라 모든 위치의 점수를 계산합니다.
        # 같은 RNN 가중치와 같은 출력 계층이 모든 시점에 사용됩니다.
        logits = self.head(states)
        # 손실 계산에 원래 점수가 필요하므로 forward 안에서는 Softmax를 적용하지 않습니다.
        return logits

model_a = RNNLanguageModel(len(vocab_a))


### A-4. 출력 Shape와 손실 함수 확인

두 문장을 모델에 넣고 Embedding, Hidden State, Logits의 Shape를 비교합니다. 각 위치의 점수를 정답과 비교하는 Cross-Entropy 입력도 확인합니다.

In [ ]:
# 두 문장만 꺼내 각 연산의 출력 형태를 확인합니다. 이 셀은 학습하지 않습니다.
with torch.no_grad():
    embedded = model_a.embedding(x_a[:2])
    states, h_n = model_a.rnn(embedded)
    logits = model_a.head(states)

print("토큰 ID:", tuple(x_a[:2].shape))
print("Embedding:", tuple(embedded.shape))
print("모든 Hidden State:", tuple(states.shape))
print("마지막 Hidden State:", tuple(h_n.shape))
print("어휘별 Logits:", tuple(logits.shape))

# (2, 5, V)의 앞 두 축만 합쳐 10개 예측 위치를 만들고, 어휘 축 V는 그대로 둡니다.
# 정답도 같은 순서로 (2, 5) -> (10,)으로 펼쳐 위치별 대응을 유지합니다.
flat_logits = logits.reshape(-1, logits.size(-1))
flat_targets = y_a[:2].reshape(-1)
loss_fn_a = nn.CrossEntropyLoss()
loss = loss_fn_a(flat_logits, flat_targets)
print("손실 함수 입력:", tuple(flat_logits.shape), tuple(flat_targets.shape))
print(f"학습 전 두 문장의 손실: {loss.item():.4f}")


### A-5. 모델 학습

앞에서 만든 입력·정답을 사용해 순전파, 손실 계산, 역전파, 가중치 갱신을 반복합니다. 출력되는 평균 손실이 어떻게 달라지는지 확인합니다.

In [ ]:
# Embedding, RNN, Linear의 학습 가능한 파라미터를 모두 Optimizer에 등록합니다.
optimizer_a = torch.optim.Adam(model_a.parameters(), lr=0.01)
history_a = []

for epoch in range(1, 81):
    model_a.train()  # 학습 모드로 전환합니다. 이 호출만으로 가중치가 바뀌지는 않습니다.
    loss_sum = 0.0
    token_count = 0

    for batch_x, batch_y in loader_a:
        # 이전 배치의 기울기가 이번 배치에 누적되지 않도록 초기화합니다.
        optimizer_a.zero_grad()
        logits = model_a(batch_x)
        # Softmax 없이 Logits를 전달합니다. 위치별 정답은 정수 토큰 ID입니다.
        loss = loss_fn_a(
            logits.reshape(-1, logits.size(-1)),
            batch_y.reshape(-1),
        )
        # 시간축으로 연결된 RNN 계산까지 역전파하여 공유 가중치의 기울기를 구합니다.
        loss.backward()
        # 계산한 기울기를 사용해 실제 파라미터를 갱신합니다.
        optimizer_a.step()

        # loss는 배치 안의 토큰 평균입니다. 토큰 수를 곱해 합계로 모읍니다.
        # 마지막 배치가 작더라도, 전체 합 / 전체 토큰 수로 평균을 올바르게 계산합니다.
        loss_sum += loss.item() * batch_y.numel()
        token_count += batch_y.numel()

    epoch_loss = loss_sum / token_count
    history_a.append(epoch_loss)
    if epoch == 1 or epoch % 20 == 0:
        print(f"epoch {epoch:2d} | 평균 손실 {epoch_loss:.4f}")


### A-6. 이전 문맥에 따른 다음 토큰 예측

학습한 모델에 주어가 다른 네 문장을 입력해, '항상' 다음 토큰의 상위 후보와 확률을 비교합니다. 문장 끝에서 EOS를 예측하는지도 확인합니다.

In [ ]:
def predict_next_token(model, text: str, token_to_id: dict, vocab: list, top_k: int = 3):
    # 문맥 전체를 ID로 바꾸고, 문장 한 개를 나타내는 배치 축을 추가합니다.
    token_ids = torch.tensor(
        [[token_to_id[token] for token in text.split()]], dtype=torch.long
    )
    model.eval()
    # 추론에서는 기울기를 계산하거나 가중치를 갱신하지 않습니다.
    with torch.no_grad():
        logits = model(token_ids)
        # 마지막 입력 토큰까지 읽은 위치의 점수가 '그다음 토큰'의 예측입니다.
        last_logits = logits[0, -1, :]
        # 어휘 축에 Softmax를 적용해 확률로 표시합니다. 학습 손실 계산과는 구분합니다.
        probabilities = torch.softmax(last_logits, dim=-1)
        top_probs, top_ids = torch.topk(probabilities, k=top_k)
    return [(vocab[index], prob) for index, prob in zip(top_ids.tolist(), top_probs.tolist())]

# 마지막 입력 토큰은 모두 '항상'이지만, 앞의 주어를 바꾸어 결과를 비교합니다.
for text in ["나는 아침에 항상", "너는 아침에 항상", "우리는 아침에 항상", "친구는 아침에 항상"]:
    candidates = predict_next_token(model_a, text, token_to_id_a, vocab_a)
    result = ", ".join(f"{token} {prob:.1%}" for token, prob in candidates)
    print(f"{text} -> {result}")

# 문장의 마지막 단어 다음에는 종료 토큰이 예측되는지도 확인합니다.
print("문장 끝:", predict_next_token(model_a, "나는 아침에 항상 커피를 마신다", token_to_id_a, vocab_a, 1))


## B형 1단계 · A형 예제 응용

다른 학습 문장에 A형의 입력·정답 Shift와 순전파를 적용합니다. **아래 두 코드 셀의 `[학생 작성]` 부분만 완성**하세요.

| 작성 기능 | 요구 결과 |
|---|---|
| `make_lm_xy(encoded)` | 원본 `(12, 6)` → 입력·정답 각각 `(12, 5)` |
| `StudentRNNLanguageModel.forward(token_ids)` | 입력 `(B, T)` → 모든 시점의 Logits `(B, T, 어휘 수)` |

첫 문장의 입력은 `[학생은, 점심에, 항상, 수학을, 공부한다]`, 정답은 `[점심에, 항상, 수학을, 공부한다, <EOS>]`가 되어야 합니다.

### B-1. 새 문장 데이터 준비

학생·선생님·작가·화가가 하는 활동을 표현한 문장 12개를 토큰 ID로 변환합니다. **이 셀은 수정하지 않습니다.**

In [ ]:
# 제공 데이터: 새 어휘집으로 다시 구성합니다. 각 문장은 <EOS>를 붙이기 전 5개 토큰입니다.
sentences_b = [
    "학생은 점심에 항상 수학을 공부한다",
    "학생은 저녁에 항상 수학을 공부한다",
    "학생은 밤에 항상 수학을 공부한다",
    "선생님은 아침에 항상 수업을 준비한다",
    "선생님은 저녁에 항상 수업을 준비한다",
    "선생님은 밤에 항상 수업을 준비한다",
    "작가는 아침에 항상 소설을 쓴다",
    "작가는 점심에 항상 소설을 쓴다",
    "작가는 밤에 항상 소설을 쓴다",
    "화가는 아침에 항상 그림을 그린다",
    "화가는 점심에 항상 그림을 그린다",
    "화가는 저녁에 항상 그림을 그린다",
]
tokens_b = [sentence.split() + ["<EOS>"] for sentence in sentences_b]
vocab_b = sorted({token for sentence in tokens_b for token in sentence})
token_to_id_b = {token: index for index, token in enumerate(vocab_b)}
encoded_b = torch.tensor(
    [[token_to_id_b[token] for token in sentence] for sentence in tokens_b],
    dtype=torch.long,
)

# 학습 문장에는 없는 주어·시간 조합입니다. 단어 자체는 모두 학습 어휘집에 있습니다.
# 아래 확인 문장과 기대 토큰은 학습 데이터에 추가하지 않습니다.
check_prompts_b = [
    ("학생은 아침에 항상", "수학을"),
    ("선생님은 점심에 항상", "수업을"),
    ("작가는 저녁에 항상", "소설을"),
    ("화가는 밤에 항상", "그림을"),
]
print("학습 문장 수 / 어휘 수:", len(sentences_b), len(vocab_b))
print("원본 ID Shape:", tuple(encoded_b.shape))


### B-2. [학생 작성] 입력·정답 Shift 함수

A형과 같이 마지막 토큰을 제외한 입력, 첫 토큰을 제외한 정답을 반환하도록 함수를 완성합니다.

In [ ]:
def make_lm_xy(encoded: torch.Tensor):
    # TODO: 입력과 정답을 한 위치 어긋나게 구성해 (inputs, targets)로 반환하세요.
    raise NotImplementedError("B형 1단계: make_lm_xy를 완성하세요.")


### B-3. [학생 작성] RNN 순전파

제공된 Embedding·RNN·Linear 계층을 연결해 모든 시점의 Logits를 반환합니다. Softmax는 적용하지 않습니다.

In [ ]:
class StudentRNNLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 16, hidden_size: int = 32):
        super().__init__()
        # 계층 정의는 제공합니다. forward에서 세 계층을 연결하세요.
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, vocab_size)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        # TODO: 모든 시점의 어휘별 Logits를 반환하세요.
        raise NotImplementedError("B형 1단계: forward를 완성하세요.")


### B-4. B형 1단계 결과 확인 (제공 코드)

입력·정답의 토큰 대응과 순전파 출력 Shape를 확인합니다. **이 셀은 수정하지 않습니다.**

In [ ]:
# 제공 확인 코드: B형 1단계를 완성한 뒤 실행합니다.
x_b, y_b = make_lm_xy(encoded_b)
first_input = [vocab_b[index] for index in x_b[0].tolist()]
first_target = [vocab_b[index] for index in y_b[0].tolist()]
assert x_b.shape == y_b.shape == (12, 5), "입력과 정답의 Shape를 확인하세요."
assert first_input == ["학생은", "점심에", "항상", "수학을", "공부한다"], "입력의 범위를 확인하세요."
assert first_target == ["점심에", "항상", "수학을", "공부한다", "<EOS>"], "정답의 Shift를 확인하세요."

# B형 모델은 A형과 다른 어휘집을 사용하므로 새로 만듭니다.
torch.manual_seed(42)
model_b = StudentRNNLanguageModel(len(vocab_b))
with torch.no_grad():
    sample_logits = model_b(x_b[:2])
assert sample_logits.shape == (2, 5, len(vocab_b)), "마지막 상태만 사용하지 않았는지 확인하세요."
print("입력:", first_input)
print("정답:", first_target)
print("Logits Shape:", tuple(sample_logits.shape))
print("B형 1단계 통과")


## B형 2단계 · 시간·날씨 조건에 따른 활동 예측

**새 문제:** 주어가 아니라 **시간과 날씨 두 조건**을 보고 다음 활동을 예측합니다. 모델은 이 규칙을 이미 학습한 상태이므로 **다시 학습하지 않습니다.**

| 입력 문맥 | 기대하는 다음 토큰 |
|---|---|
| 아침 맑음 오늘 추천은 | 산책 |
| 아침 비 오늘 추천은 | 독서 |
| 저녁 맑음 오늘 추천은 | 운동 |
| 저녁 비 오늘 추천은 | 영화 |

**[학생 작성] `evaluate_conditions()` 함수 하나**를 완성하세요. A형의 `predict_next_token()`을 호출하지 않고, 여러 문장을 **한 번의 Batch 추론**으로 처리합니다.

1. 입력 문장 4개를 토큰 ID로 변환하여 `(4, 4)` Tensor로 만듭니다.
2. 모델을 한 번 실행하고 **각 문장의 마지막 위치 Logits**만 선택합니다.
3. Softmax로 어휘별 확률을 구하고, 가장 높은 확률의 토큰 ID와 그 확률을 얻습니다.
4. ID를 토큰 문자열로 바꾸고, 기대 정답과 비교해 **정확도(맞힌 개수 / 전체 개수)**를 계산합니다.
5. 문장별 `(입력, 예측 토큰, 예측 확률, 정답, 일치 여부)`와 전체 정확도를 반환합니다.

같은 시간이라도 날씨에 따라, 같은 날씨라도 시간에 따라 예측이 달라지는지 확인하세요.

### B-5. 새 모델과 검증 문장 준비 (제공 코드)

시간·날씨 규칙을 학습한 RNN 가중치를 불러옵니다. 이 모델은 A형과 같은 구조지만 **어휘집과 학습 데이터가 다릅니다.** 이 셀은 수정하지 않습니다.

In [ ]:
from pathlib import Path

# 제공된 가중치는 현재 노트북과 같은 폴더의 weights 폴더에 있습니다.
checkpoint = torch.load(Path("weights") / "time_weather_rnn.pt", map_location="cpu", weights_only=True)
vocab_c = checkpoint["vocab"]
token_to_id_c = {token: index for index, token in enumerate(vocab_c)}

# A형에서 사용한 것과 같은 RNN 구조로 모델을 만들고 학습된 가중치를 불러옵니다.
model_c = RNNLanguageModel(len(vocab_c))
model_c.load_state_dict(checkpoint["model_state"])
model_c.eval()

# 네 문장은 모두 입력 길이가 4로 같아 하나의 Batch로 묶을 수 있습니다.
condition_prompts = [
    "아침 맑음 오늘 추천은",
    "아침 비 오늘 추천은",
    "저녁 맑음 오늘 추천은",
    "저녁 비 오늘 추천은",
]
condition_targets = ["산책", "독서", "운동", "영화"]

print("비교할 문장 수:", len(condition_prompts))
print("모델 어휘 수:", len(vocab_c))

### B-6. [학생 작성] Batch 예측·평가 함수

**아래 함수 본문만 직접 작성합니다.** A형은 문장 하나의 Top-k 후보를 반환했지만, 여기서는 네 문장을 한 번에 처리하고 정답과 비교해야 합니다. 반환값의 형식은 함수 아래 주석을 참고하세요.

In [ ]:
def evaluate_conditions(model, prompts, targets, token_to_id, vocab):
    """입력 문장 여러 개를 한 번에 예측하고, 문장별 결과와 정확도를 반환합니다.

    반환 형식:
        rows: [(입력문장, 예측토큰, 예측확률(float), 정답, 일치여부(bool)), ...]
        accuracy: 정확도(float, 0~1)
    """
    # ===== [학생 작성 시작] B형 2단계: 아래 함수를 직접 구현하세요. =====
    # 1) prompts 전체를 토큰 ID로 변환 -> (문장 수, 시퀀스 길이) Tensor
    # 2) model.eval()과 torch.no_grad()로 Batch 추론
    # 3) 모든 시점의 Logits 중 마지막 위치 선택 -> (문장 수, 어휘 수)
    # 4) Softmax, argmax로 예측 토큰과 확률 계산
    # 5) targets와 비교한 rows 및 accuracy 만들기
    raise NotImplementedError("B형 2단계: evaluate_conditions()를 완성하세요.")
    # ===== [학생 작성 끝] =====

### B-7. 예측 결과 확인 (제공 코드)

작성한 함수의 반환값으로 네 조건의 예측 토큰·확률·정답을 출력합니다. **이 셀은 수정하지 않습니다.**

In [ ]:
rows, accuracy = evaluate_conditions(
    model_c, condition_prompts, condition_targets, token_to_id_c, vocab_c
)

for prompt, predicted, probability, target, matched in rows:
    print(f"{prompt} → {predicted} ({probability:.1%}) | 정답: {target} | {'O' if matched else 'X'}")
print(f"정확도: {accuracy:.0%} ({sum(row[4] for row in rows)}/{len(rows)})")

# 제공된 모델과 입력 조건이 올바르게 연결되었는지 확인합니다.
assert len(rows) == 4, "네 문장 모두 평가했는지 확인하세요."
assert 0 <= accuracy <= 1, "정확도를 0~1 사이 값으로 반환하세요."

### 결과 확인

- A형: 마지막 토큰이 같아도 앞 주어에 따라 예측이 달라졌는지 확인합니다.
- B형 1단계: 입력·정답 Shift와 모든 시점의 Logits Shape를 확인합니다.
- B형 2단계: 시간과 날씨의 **두 조건이 함께** 바뀔 때 예측 결과를 비교하고, 네 문장의 정확도를 확인합니다.